# Notebook 3: Evaluation and Robustness Analysis

**Purpose:** Evaluate all 11 models and 55 pairwise ensembles on EyePACS and APTOS.

Produces all results for the NPJ Digital Medicine manuscript:
1. Comprehensive OR-gate evaluation (all 55 pairs × 2 datasets)
2. Single-model threshold sweep
3. 2×2 factorial decomposition (rule effect vs. metric effect)
4. Severity-weight robustness analysis
5. Bootstrap confidence intervals (n=1,000)

**Requirements:** Trained models from Notebook 2 on Google Drive.

**Time:** ~30 minutes (dominated by model inference and bootstrap).

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q timm transformers

from huggingface_hub import login
login()  # Enter your HuggingFace token when prompted

## Part 1: Comprehensive Evaluation

Load all 11 models, generate predictions on both datasets, compute OR-gate analysis for all 55 pairs.

In [ ]:
# ============================================================
# COMPREHENSIVE EVALUATION: 11 Models × 2 Datasets
# EyePACS (internal) + APTOS (external validation)
# OR-gate severity-weighted analysis
# ============================================================

import torch, torch.nn as nn, timm
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as transforms
from itertools import combinations
import json, os, warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

drive_base = '/content/drive/MyDrive/stochastic_orthogonality'
FULL_DIR = f'{drive_base}/data/eyepacs_full'
EYEPACS_PROCESSED = f'{FULL_DIR}/processed'
APTOS_PROCESSED = f'{drive_base}/data/aptos/processed'
MODEL_DIR = f'{drive_base}/models_eyepacs_full'
RESULTS_DIR = f'{drive_base}/results_eyepacs_full'
os.makedirs(RESULTS_DIR, exist_ok=True)

val_transform = transforms.Compose([
    transforms.Resize(256), transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Dataset class that works for both EyePACS and APTOS
class DRDataset(Dataset):
    def __init__(self, csv_path, image_dir, transform, id_col='image_id', grade_col='level'):
        self.df = pd.read_csv(csv_path)
        self.image_dir = image_dir
        self.transform = transform
        self.id_col = id_col
        self.grade_col = grade_col
        if 'binary_label' not in self.df.columns:
            self.df['binary_label'] = (self.df[grade_col] >= 2).astype(int)
        existing = set(os.path.splitext(f)[0] for f in os.listdir(image_dir) if f.endswith('.png'))
        self.df = self.df[self.df[id_col].isin(existing)].reset_index(drop=True)
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(os.path.join(self.image_dir, f"{row[self.id_col]}.png")).convert('RGB')
        if self.transform: img = self.transform(img)
        return img, torch.tensor(row['binary_label'], dtype=torch.float32), str(row[self.id_col])

# Load test sets
print('Loading test sets...')
eyepacs_test = DRDataset(f'{FULL_DIR}/eyepacs_full_test.csv', EYEPACS_PROCESSED, val_transform)
eyepacs_loader = DataLoader(eyepacs_test, batch_size=32, shuffle=False, num_workers=2)
print(f'  EyePACS test: {len(eyepacs_test)} images')

# APTOS — use ALL data as external test (models never trained on it)
aptos_raw = pd.read_csv(f'{drive_base}/data/aptos/raw/train.csv')
aptos_raw.columns = ['image_id', 'level']
aptos_raw['binary_label'] = (aptos_raw['level'] >= 2).astype(int)
aptos_csv_path = f'{drive_base}/data/aptos/aptos_external_test.csv'
aptos_raw[['image_id', 'level', 'binary_label']].to_csv(aptos_csv_path, index=False)

aptos_test = DRDataset(aptos_csv_path, APTOS_PROCESSED, val_transform)
aptos_loader = DataLoader(aptos_test, batch_size=32, shuffle=False, num_workers=2)
print(f'  APTOS external test: {len(aptos_test)} images')

# Grade distributions
for name, ds in [('EyePACS', eyepacs_test), ('APTOS', aptos_test)]:
    grades = ds.df['level'].value_counts().sort_index()
    severe = (ds.df['level'] >= 3).sum()
    print(f'  {name} grades: {grades.to_dict()}, Severe: {severe}')

# ============================================================
# Load all 11 models and get predictions on both datasets
# ============================================================
print(f'\n{"="*70}')
print('LOADING MODELS AND GENERATING PREDICTIONS')
print(f'{"="*70}')

from transformers import AutoModel

class RETFoundBinary(nn.Module):
    def __init__(self, base):
        super().__init__()
        self.base = base
        self.classifier = nn.Sequential(nn.LayerNorm(1024), nn.Linear(1024, 1))
    def forward(self, x):
        return self.classifier(self.base(x).last_hidden_state[:, 0])

class RETFound5Class(nn.Module):
    def __init__(self, base):
        super().__init__()
        self.base = base
        self.classifier = nn.Sequential(nn.LayerNorm(1024), nn.Dropout(0.1), nn.Linear(1024, 5))
    def forward(self, x):
        return self.classifier(self.base(x).last_hidden_state[:, 0])

def get_preds(model, loader, is_5class=False):
    model.eval()
    probs, labs, ids = [], [], []
    with torch.no_grad():
        for images, labels, img_ids in loader:
            images = images.to(device)
            if is_5class:
                logits = model(images)
                p5 = torch.softmax(logits, dim=-1)
                p = p5[:, 2] + p5[:, 3] + p5[:, 4]
            else:
                p = torch.sigmoid(model(images).squeeze(-1))
            probs.extend(p.cpu().numpy())
            labs.extend(labels.numpy())
            ids.extend(img_ids)
    return np.array(probs), np.array(labs), ids

# Model definitions
MODEL_DEFS = [
    ('densenet121_binary', 'densenet121', 'binary', 1),
    ('densenet121_5class', 'densenet121', '5class', 5),
    ('efficientnet_b3_binary', 'efficientnet_b3', 'binary', 1),
    ('efficientnet_b3_5class', 'efficientnet_b3', '5class', 5),
    ('resnet50_binary', 'resnet50', 'binary', 1),
    ('resnet50_5class', 'resnet50', '5class', 5),
    ('vit_base_binary', 'vit_base_patch16_224', 'binary', 1),
    ('vit_base_5class', 'vit_base_patch16_224', '5class', 5),
    ('retfound_binary', 'retfound', 'binary', 1),
    ('retfound_5class', 'retfound', '5class', 5),
    ('retfound_adversarial', 'retfound', 'binary', 1),
]

eyepacs_preds = {}
aptos_preds = {}

for save_name, arch, task, nc in MODEL_DEFS:
    model_path = f'{MODEL_DIR}/{save_name}.pt'
    if not os.path.exists(model_path):
        print(f'  MISSING: {save_name}')
        continue

    print(f'\n  Loading {save_name}...')

    if arch == 'retfound':
        base = AutoModel.from_pretrained("iszt/RETFound_mae_meh")
        if nc == 5:
            model = RETFound5Class(base).to(device)
        else:
            model = RETFoundBinary(base).to(device)
    else:
        model = timm.create_model(arch, pretrained=False, num_classes=nc).to(device)

    model.load_state_dict(torch.load(model_path, map_location=device))
    is_5c = (task == '5class')

    probs_e, labs_e, ids_e = get_preds(model, eyepacs_loader, is_5c)
    auc_e = roc_auc_score(labs_e, probs_e)
    eyepacs_preds[save_name] = {'probs': probs_e, 'labels': labs_e, 'ids': ids_e, 'auc': auc_e}

    probs_a, labs_a, ids_a = get_preds(model, aptos_loader, is_5c)
    auc_a = roc_auc_score(labs_a, probs_a)
    aptos_preds[save_name] = {'probs': probs_a, 'labels': labs_a, 'ids': ids_a, 'auc': auc_a}

    print(f'    EyePACS AUC: {auc_e:.4f}, APTOS AUC: {auc_a:.4f}')

    del model; torch.cuda.empty_cache()

# ============================================================
# OR-GATE ANALYSIS FUNCTION
# ============================================================

SEVERITY_WEIGHTS = {4: 500, 3: 200, 2: 15, 1: 2, 'fp': 1}

def or_gate_analysis(preds_dict, dataset_df, dataset_name, id_col='image_id', grade_col='level'):
    print(f'\n{"="*90}')
    print(f'OR-GATE ANALYSIS: {dataset_name}')
    print(f'{"="*90}')

    model_names = list(preds_dict.keys())
    labels = preds_dict[model_names[0]]['labels']
    ids = preds_dict[model_names[0]]['ids']

    id_to_grade = dict(zip(dataset_df[id_col].astype(str), dataset_df[grade_col]))
    grades = np.array([id_to_grade.get(str(tid), -1) for tid in ids])

    severe_mask = grades >= 3
    severe_indices = np.where(severe_mask)[0]

    print(f'\n  Test set: {len(labels)} images')
    print(f'  Severe (grade 3+4): {severe_mask.sum()}')
    for g in range(5):
        print(f'    Grade {g}: {(grades == g).sum()}')

    # Individual models
    print(f'\n  {"Model":<30s} {"AUC":<8s} {"Sev Miss":<10s} {"G4 Miss":<8s} {"G3 Miss":<8s}')
    print(f'  {"-"*64}')

    model_catches = {}
    model_misses = {}
    model_costs = {}

    for name in model_names:
        probs = preds_dict[name]['probs']
        preds = (probs >= 0.5).astype(int)
        auc = preds_dict[name]['auc']

        catches, misses = set(), set()
        for idx in severe_indices:
            if labels[idx] == 1:
                if preds[idx] == 1: catches.add(idx)
                else: misses.add(idx)
        model_catches[name] = catches
        model_misses[name] = misses

        g4m = sum(1 for idx in misses if grades[idx] == 4)
        g3m = sum(1 for idx in misses if grades[idx] == 3)

        cost = 0
        for i in range(len(preds)):
            if preds[i] != labels[i]:
                if labels[i] == 1: cost += SEVERITY_WEIGHTS.get(grades[i], 2)
                else: cost += SEVERITY_WEIGHTS['fp']
        model_costs[name] = cost

        print(f'  {name:<30s} {auc:<8.4f} {len(misses):<10d} {g4m:<8d} {g3m:<8d}')

    best_name = min(model_costs, key=model_costs.get)
    best_cost = model_costs[best_name]
    best_misses = model_misses[best_name]

    # Pairwise OR-gate
    print(f'\n  Best single model: {best_name} (cost={best_cost})')

    print(f'\n  {"Ensemble":<50s} {"BothMiss":<10s} {"G4 BM":<8s} {"G3 BM":<8s} '
          f'{"OR Cost":<10s} {"vs Best":<10s} {"rho":<8s} {"Type":<6s}')
    print(f'  {"-"*112}')

    or_results = []

    for i, n1 in enumerate(model_names):
        for j, n2 in enumerate(model_names):
            if j <= i: continue

            e1 = ((preds_dict[n1]['probs'] >= 0.5).astype(int) != labels).astype(int)
            e2 = ((preds_dict[n2]['probs'] >= 0.5).astype(int) != labels).astype(int)
            rho = np.corrcoef(e1, e2)[0, 1] if e1.std() > 0 and e2.std() > 0 else np.nan

            both_miss = model_misses[n1] & model_misses[n2]
            g4_bm = sum(1 for idx in both_miss if grades[idx] == 4)
            g3_bm = sum(1 for idx in both_miss if grades[idx] == 3)

            p1 = (preds_dict[n1]['probs'] >= 0.5).astype(int)
            p2 = (preds_dict[n2]['probs'] >= 0.5).astype(int)
            or_preds = np.maximum(p1, p2)

            or_cost = 0
            for idx in range(len(or_preds)):
                if or_preds[idx] != labels[idx]:
                    if labels[idx] == 1: or_cost += SEVERITY_WEIGHTS.get(grades[idx], 2)
                    else: or_cost += SEVERITY_WEIGHTS['fp']

            vs_best = or_cost - best_cost

            arch1 = n1.split('_')[0] if 'retfound' not in n1 else 'retfound'
            arch2 = n2.split('_')[0] if 'retfound' not in n2 else 'retfound'
            task1 = '5class' if '5class' in n1 else 'binary'
            task2 = '5class' if '5class' in n2 else 'binary'
            same_arch = (arch1 == arch2)
            same_task = (task1 == task2)

            if same_arch and not same_task: div = 'TASK'
            elif not same_arch and same_task: div = 'ARCH'
            elif not same_arch and not same_task: div = 'BOTH'
            else: div = 'SAME'

            result = {
                'n1': n1, 'n2': n2, 'div': div, 'rho': float(rho),
                'both_miss': len(both_miss), 'g4_bm': g4_bm, 'g3_bm': g3_bm,
                'or_cost': or_cost, 'vs_best': vs_best
            }
            or_results.append(result)

            print(f'  {n1+" + "+n2:<50s} {len(both_miss):<10d} {g4_bm:<8d} {g3_bm:<8d} '
                  f'{or_cost:<10d} {vs_best:<+10d} {rho:<8.4f} {div:<6s}')

    # Rankings
    print(f'\n  TOP 10 BY RESIDUAL RISK (lowest Both Miss):')
    for rank, r in enumerate(sorted(or_results, key=lambda x: (x['g4_bm'], x['g3_bm'], x['both_miss']))[:10]):
        print(f'    {rank+1}. {r["n1"]+" + "+r["n2"]:<50s} BM={r["both_miss"]} '
              f'G4={r["g4_bm"]} G3={r["g3_bm"]} rho={r["rho"]:.4f} ({r["div"]})')

    print(f'\n  TOP 10 BY OR-GATE COST (lowest = safest):')
    for rank, r in enumerate(sorted(or_results, key=lambda x: x['or_cost'])[:10]):
        print(f'    {rank+1}. {r["n1"]+" + "+r["n2"]:<50s} Cost={r["or_cost"]} '
              f'vs_best={r["vs_best"]:+d} rho={r["rho"]:.4f} ({r["div"]})')

    # By diversity type
    print(f'\n  BY DIVERSITY TYPE:')
    for div in ['TASK', 'ARCH', 'BOTH', 'SAME']:
        subset = [r for r in or_results if r['div'] == div]
        if not subset: continue
        print(f'    {div} ({len(subset)} pairs): Avg BothMiss={np.mean([r["both_miss"] for r in subset]):.1f}, '
              f'Avg Cost={np.mean([r["or_cost"] for r in subset]):.0f}, '
              f'Avg rho={np.nanmean([r["rho"] for r in subset]):.4f}')

    # Rho correlation
    rhos = [r['rho'] for r in or_results if not np.isnan(r['rho'])]
    bms = [r['both_miss'] for r in or_results if not np.isnan(r['rho'])]
    costs = [r['or_cost'] for r in or_results if not np.isnan(r['rho'])]
    if len(rhos) > 2:
        r_bm = np.corrcoef(rhos, bms)[0, 1]
        r_cost = np.corrcoef(rhos, costs)[0, 1]
        print(f'\n  CORRELATION WITH RHO:')
        print(f'    rho vs Both Miss: r={r_bm:.4f}')
        print(f'    rho vs OR-gate Cost: r={r_cost:.4f}')

    return or_results

# ============================================================
# RUN ON EYEPACS
# ============================================================
eyepacs_df = pd.read_csv(f'{FULL_DIR}/eyepacs_full_test.csv')
eyepacs_results = or_gate_analysis(eyepacs_preds, eyepacs_df, 'EYEPACS (Internal)')

# ============================================================
# RUN ON APTOS
# ============================================================
aptos_df = pd.read_csv(aptos_csv_path)
aptos_results = or_gate_analysis(aptos_preds, aptos_df, 'APTOS (External Validation)')

# ============================================================
# CROSS-DATASET COMPARISON
# ============================================================
print(f'\n{"="*90}')
print('CROSS-DATASET COMPARISON: Does the finding replicate?')
print(f'{"="*90}')

print(f'\n  {"Metric":<45s} {"EyePACS":<20s} {"APTOS":<20s}')
print(f'  {"-"*85}')

best_e_name = max(eyepacs_preds.keys(), key=lambda k: eyepacs_preds[k]['auc'])
best_a_name = max(aptos_preds.keys(), key=lambda k: aptos_preds[k]['auc'])
print(f'  {"Best single model":<45s} {best_e_name:<20s} {best_a_name:<20s}')
print(f'  {"Best single AUC":<45s} {eyepacs_preds[best_e_name]["auc"]:<20.4f} {aptos_preds[best_a_name]["auc"]:<20.4f}')

best_e_or = sorted(eyepacs_results, key=lambda x: x['or_cost'])[0]
best_a_or = sorted(aptos_results, key=lambda x: x['or_cost'])[0]
print(f'  {"Best OR-gate pair":<45s} {(best_e_or["n1"][:15]+"+"+best_e_or["n2"][:15]):<20s} {(best_a_or["n1"][:15]+"+"+best_a_or["n2"][:15]):<20s}')
print(f'  {"OR-gate cost reduction":<45s} {best_e_or["vs_best"]:<+20d} {best_a_or["vs_best"]:<+20d}')

e_rhos = [r['rho'] for r in eyepacs_results if not np.isnan(r['rho'])]
e_bms = [r['both_miss'] for r in eyepacs_results if not np.isnan(r['rho'])]
a_rhos = [r['rho'] for r in aptos_results if not np.isnan(r['rho'])]
a_bms = [r['both_miss'] for r in aptos_results if not np.isnan(r['rho'])]
e_corr = np.corrcoef(e_rhos, e_bms)[0, 1]
a_corr = np.corrcoef(a_rhos, a_bms)[0, 1]
print(f'  {"rho vs BothMiss correlation":<45s} {e_corr:<20.4f} {a_corr:<20.4f}')

e_rhos_c = [r['rho'] for r in eyepacs_results if not np.isnan(r['rho'])]
e_costs = [r['or_cost'] for r in eyepacs_results if not np.isnan(r['rho'])]
a_rhos_c = [r['rho'] for r in aptos_results if not np.isnan(r['rho'])]
a_costs = [r['or_cost'] for r in aptos_results if not np.isnan(r['rho'])]
e_corr_c = np.corrcoef(e_rhos_c, e_costs)[0, 1]
a_corr_c = np.corrcoef(a_rhos_c, a_costs)[0, 1]
print(f'  {"rho vs OR-gate Cost correlation":<45s} {e_corr_c:<20.4f} {a_corr_c:<20.4f}')

# Save everything
final_results = {
    'eyepacs': {
        'n_test': len(eyepacs_test),
        'model_aucs': {k: float(v['auc']) for k, v in eyepacs_preds.items()},
        'or_gate_results': eyepacs_results,
        'rho_bothmiss_corr': float(e_corr),
        'rho_cost_corr': float(e_corr_c)
    },
    'aptos': {
        'n_test': len(aptos_test),
        'model_aucs': {k: float(v['auc']) for k, v in aptos_preds.items()},
        'or_gate_results': aptos_results,
        'rho_bothmiss_corr': float(a_corr),
        'rho_cost_corr': float(a_corr_c)
    },
    'severity_weights': {str(k): v for k, v in SEVERITY_WEIGHTS.items()}
}

with open(f'{RESULTS_DIR}/comprehensive_evaluation.json', 'w') as f:
    json.dump(final_results, f, indent=2, default=str)
print(f'\nAll results saved to {RESULTS_DIR}/comprehensive_evaluation.json')

## Part 2: Load Predictions for Robustness Analyses

Reloads predictions and defines the severity-cost function used by all subsequent analyses.

In [ ]:
# ============================================================
# CELL 1: Load all model predictions on both datasets
# (Needed for threshold sweep, 2x2, and robustness analyses)
# ============================================================

import torch, torch.nn as nn, timm
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as transforms
import json, os, warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

drive_base = '/content/drive/MyDrive/stochastic_orthogonality'
FULL_DIR = f'{drive_base}/data/eyepacs_full'
EYEPACS_PROCESSED = f'{FULL_DIR}/processed'
APTOS_PROCESSED = f'{drive_base}/data/aptos/processed'
MODEL_DIR = f'{drive_base}/models_eyepacs_full'
RESULTS_DIR = f'{drive_base}/results_eyepacs_full'

val_transform = transforms.Compose([
    transforms.Resize(256), transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

class DRDataset(Dataset):
    def __init__(self, csv_path, image_dir, transform):
        self.df = pd.read_csv(csv_path)
        self.image_dir = image_dir
        self.transform = transform
        if 'binary_label' not in self.df.columns:
            self.df['binary_label'] = (self.df['level'] >= 2).astype(int)
        existing = set(os.path.splitext(f)[0] for f in os.listdir(image_dir) if f.endswith('.png'))
        self.df = self.df[self.df['image_id'].isin(existing)].reset_index(drop=True)
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(os.path.join(self.image_dir, f"{row['image_id']}.png")).convert('RGB')
        if self.transform: img = self.transform(img)
        return img, torch.tensor(row['binary_label'], dtype=torch.float32), str(row['image_id'])

# Load test sets
print('Loading test sets...')
eyepacs_test = DRDataset(f'{FULL_DIR}/eyepacs_full_test.csv', EYEPACS_PROCESSED, val_transform)
eyepacs_loader = DataLoader(eyepacs_test, batch_size=32, shuffle=False, num_workers=2)

aptos_raw = pd.read_csv(f'{drive_base}/data/aptos/raw/train.csv')
aptos_raw.columns = ['image_id', 'level']
aptos_raw['binary_label'] = (aptos_raw['level'] >= 2).astype(int)
aptos_csv = f'{drive_base}/data/aptos/aptos_external_test.csv'
aptos_raw[['image_id', 'level', 'binary_label']].to_csv(aptos_csv, index=False)
aptos_test = DRDataset(aptos_csv, APTOS_PROCESSED, val_transform)
aptos_loader = DataLoader(aptos_test, batch_size=32, shuffle=False, num_workers=2)

print(f'EyePACS: {len(eyepacs_test)}, APTOS: {len(aptos_test)}')

# Grade arrays
eyepacs_df = eyepacs_test.df
aptos_df = aptos_test.df
eyepacs_grades = eyepacs_df['level'].values
aptos_grades = aptos_df['level'].values

from transformers import AutoModel

class RETFoundBinary(nn.Module):
    def __init__(self, base):
        super().__init__()
        self.base = base
        self.classifier = nn.Sequential(nn.LayerNorm(1024), nn.Linear(1024, 1))
    def forward(self, x):
        return self.classifier(self.base(x).last_hidden_state[:, 0])

class RETFound5Class(nn.Module):
    def __init__(self, base):
        super().__init__()
        self.base = base
        self.classifier = nn.Sequential(nn.LayerNorm(1024), nn.Dropout(0.1), nn.Linear(1024, 5))
    def forward(self, x):
        return self.classifier(self.base(x).last_hidden_state[:, 0])

def get_preds(model, loader, is_5class=False):
    model.eval()
    probs, labs, ids = [], [], []
    with torch.no_grad():
        for images, labels, img_ids in loader:
            images = images.to(device)
            if is_5class:
                logits = model(images)
                p5 = torch.softmax(logits, dim=-1)
                p = p5[:, 2] + p5[:, 3] + p5[:, 4]
            else:
                p = torch.sigmoid(model(images).squeeze(-1))
            probs.extend(p.cpu().numpy())
            labs.extend(labels.numpy())
            ids.extend(img_ids)
    return np.array(probs), np.array(labs), ids

MODEL_DEFS = [
    ('densenet121_binary', 'densenet121', 'binary', 1),
    ('densenet121_5class', 'densenet121', '5class', 5),
    ('efficientnet_b3_binary', 'efficientnet_b3', 'binary', 1),
    ('efficientnet_b3_5class', 'efficientnet_b3', '5class', 5),
    ('resnet50_binary', 'resnet50', 'binary', 1),
    ('resnet50_5class', 'resnet50', '5class', 5),
    ('vit_base_binary', 'vit_base_patch16_224', 'binary', 1),
    ('vit_base_5class', 'vit_base_patch16_224', '5class', 5),
    ('retfound_binary', 'retfound', 'binary', 1),
    ('retfound_5class', 'retfound', '5class', 5),
    ('retfound_adversarial', 'retfound', 'binary', 1),
]

eyepacs_preds = {}
aptos_preds = {}

for save_name, arch, task, nc in MODEL_DEFS:
    model_path = f'{MODEL_DIR}/{save_name}.pt'
    if not os.path.exists(model_path):
        print(f'  MISSING: {save_name}')
        continue
    print(f'  Loading {save_name}...')
    if arch == 'retfound':
        base = AutoModel.from_pretrained("iszt/RETFound_mae_meh")
        model = (RETFound5Class(base) if nc == 5 else RETFoundBinary(base)).to(device)
    else:
        model = timm.create_model(arch, pretrained=False, num_classes=nc).to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    is_5c = (task == '5class')

    probs_e, labs_e, ids_e = get_preds(model, eyepacs_loader, is_5c)
    eyepacs_preds[save_name] = {'probs': probs_e, 'labels': labs_e, 'ids': ids_e, 'auc': roc_auc_score(labs_e, probs_e)}

    probs_a, labs_a, ids_a = get_preds(model, aptos_loader, is_5c)
    aptos_preds[save_name] = {'probs': probs_a, 'labels': labs_a, 'ids': ids_a, 'auc': roc_auc_score(labs_a, probs_a)}

    print(f'    EyePACS AUC: {eyepacs_preds[save_name]["auc"]:.4f}, APTOS AUC: {aptos_preds[save_name]["auc"]:.4f}')
    del model; torch.cuda.empty_cache()

# Store shared labels and grades
e_labels = eyepacs_preds['densenet121_binary']['labels']
a_labels = aptos_preds['densenet121_binary']['labels']

WEIGHTS = {4: 500, 3: 200, 2: 15, 1: 2, 'fp': 1}

def severity_cost(preds_binary, labels, grades, weights=WEIGHTS):
    cost = 0
    for idx in range(len(preds_binary)):
        if preds_binary[idx] != labels[idx]:
            if labels[idx] == 1:
                cost += weights.get(grades[idx], 2)
            else:
                cost += weights['fp']
    return cost

print('\nAll predictions loaded. Ready for analyses.')

## Part 3: Robustness Analyses

Addresses the four key reviewer vulnerabilities:
1. **Threshold sweep:** Can a single model at a lower threshold match the OR-gate?
2. **2×2 decomposition:** Is the benefit from the rule or the metric?
3. **Weight robustness:** Does the ranking hold under different severity weights?
4. **Bootstrap CIs:** Are the key quantities statistically significant?

In [ ]:
# ============================================================
# CELL 2: Threshold sweep, 2x2 analysis, robustness
# Addresses all four NPJ reviewer vulnerabilities
# ============================================================

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from itertools import combinations

print('='*80)
print('ANALYSIS 1: SINGLE-MODEL THRESHOLD SWEEP')
print('Can a single model at a lower threshold match the OR-gate ensemble?')
print('='*80)

# Best single model = densenet121_5class
best_name = 'densenet121_5class'
best_probs_e = eyepacs_preds[best_name]['probs']
best_probs_a = aptos_preds[best_name]['probs']

# Best OR-gate ensemble = densenet121_5class + efficientnet_b3_5class
ens_name1, ens_name2 = 'densenet121_5class', 'efficientnet_b3_5class'

thresholds = np.arange(0.05, 0.95, 0.05)

print(f'\nEyePACS threshold sweep for {best_name}:')
print(f'  {"Thresh":<8} {"Sev Cost":<10} {"G4 Miss":<8} {"G3 Miss":<8} {"Sev Miss":<10} {"FP":<8} {"Sens":<8} {"Spec":<8}')
print(f'  {"-"*70}')

sweep_results_e = []
for t in thresholds:
    preds = (best_probs_e >= t).astype(int)
    cost = severity_cost(preds, e_labels, eyepacs_grades)

    severe_mask = eyepacs_grades >= 3
    severe_missed = sum(1 for idx in range(len(preds)) if severe_mask[idx] and e_labels[idx]==1 and preds[idx]==0)
    g4_missed = sum(1 for idx in range(len(preds)) if eyepacs_grades[idx]==4 and e_labels[idx]==1 and preds[idx]==0)
    g3_missed = sum(1 for idx in range(len(preds)) if eyepacs_grades[idx]==3 and e_labels[idx]==1 and preds[idx]==0)

    tp = sum(1 for idx in range(len(preds)) if e_labels[idx]==1 and preds[idx]==1)
    fn = sum(1 for idx in range(len(preds)) if e_labels[idx]==1 and preds[idx]==0)
    tn = sum(1 for idx in range(len(preds)) if e_labels[idx]==0 and preds[idx]==0)
    fp = sum(1 for idx in range(len(preds)) if e_labels[idx]==0 and preds[idx]==1)
    sens = tp/(tp+fn) if (tp+fn) > 0 else 0
    spec = tn/(tn+fp) if (tn+fp) > 0 else 0

    sweep_results_e.append({'t': t, 'cost': cost, 'g4': g4_missed, 'g3': g3_missed,
                             'sev': severe_missed, 'fp': fp, 'sens': sens, 'spec': spec})
    print(f'  {t:<8.2f} {cost:<10d} {g4_missed:<8d} {g3_missed:<8d} {severe_missed:<10d} {fp:<8d} {sens:<8.3f} {spec:<8.3f}')

# Now compute OR-gate ensemble at 0.5
p1_e = eyepacs_preds[ens_name1]['probs']
p2_e = eyepacs_preds[ens_name2]['probs']
or_preds_e = np.maximum((p1_e >= 0.5).astype(int), (p2_e >= 0.5).astype(int))
or_cost_e = severity_cost(or_preds_e, e_labels, eyepacs_grades)
or_fp_e = sum(1 for idx in range(len(or_preds_e)) if e_labels[idx]==0 and or_preds_e[idx]==1)
or_sens_e = sum(1 for idx in range(len(or_preds_e)) if e_labels[idx]==1 and or_preds_e[idx]==1) / sum(e_labels)
or_g4_e = sum(1 for idx in range(len(or_preds_e)) if eyepacs_grades[idx]==4 and e_labels[idx]==1 and or_preds_e[idx]==0)
or_g3_e = sum(1 for idx in range(len(or_preds_e)) if eyepacs_grades[idx]==3 and e_labels[idx]==1 and or_preds_e[idx]==0)
or_sev_e = sum(1 for idx in range(len(or_preds_e)) if eyepacs_grades[idx]>=3 and e_labels[idx]==1 and or_preds_e[idx]==0)

print(f'\n  OR-gate ({ens_name1} + {ens_name2}) at t=0.5:')
print(f'  {"OR@0.5":<8} {or_cost_e:<10d} {or_g4_e:<8d} {or_g3_e:<8d} {or_sev_e:<10d} {or_fp_e:<8d} {or_sens_e:<8.3f}')

# Find: at what single-model threshold do we match the OR-gate's severe miss rate?
matching = [r for r in sweep_results_e if r['sev'] <= or_sev_e]
if matching:
    best_match = min(matching, key=lambda r: r['cost'])
    print(f'\n  Best single-model threshold matching OR-gate severe misses ({or_sev_e}):')
    print(f'  Threshold: {best_match["t"]:.2f}, Cost: {best_match["cost"]}, FP: {best_match["fp"]}')
    print(f'  OR-gate cost: {or_cost_e}, OR-gate FP: {or_fp_e}')
    print(f'  Ensemble advantage: cost {best_match["cost"] - or_cost_e:+d}, FP {best_match["fp"] - or_fp_e:+d}')
else:
    print(f'\n  No single-model threshold achieves {or_sev_e} or fewer severe misses!')

# APTOS threshold sweep
print(f'\nAPTOS threshold sweep for {best_name}:')
best_probs_a = aptos_preds[best_name]['probs']
sweep_results_a = []
for t in thresholds:
    preds = (best_probs_a >= t).astype(int)
    cost = severity_cost(preds, a_labels, aptos_grades)
    g4_missed = sum(1 for idx in range(len(preds)) if aptos_grades[idx]==4 and a_labels[idx]==1 and preds[idx]==0)
    g3_missed = sum(1 for idx in range(len(preds)) if aptos_grades[idx]==3 and a_labels[idx]==1 and preds[idx]==0)
    sev_missed = sum(1 for idx in range(len(preds)) if aptos_grades[idx]>=3 and a_labels[idx]==1 and preds[idx]==0)
    fp = sum(1 for idx in range(len(preds)) if a_labels[idx]==0 and preds[idx]==1)
    sweep_results_a.append({'t': t, 'cost': cost, 'g4': g4_missed, 'g3': g3_missed, 'sev': sev_missed, 'fp': fp})
    if t in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6]:
        print(f'  t={t:.2f}: Cost={cost}, G4={g4_missed}, G3={g3_missed}, SevMiss={sev_missed}, FP={fp}')

# APTOS OR-gate
p1_a = aptos_preds['densenet121_binary']['probs']
p2_a = aptos_preds['efficientnet_b3_binary']['probs']
or_preds_a = np.maximum((p1_a >= 0.5).astype(int), (p2_a >= 0.5).astype(int))
or_cost_a = severity_cost(or_preds_a, a_labels, aptos_grades)
or_sev_a = sum(1 for idx in range(len(or_preds_a)) if aptos_grades[idx]>=3 and a_labels[idx]==1 and or_preds_a[idx]==0)
or_fp_a = sum(1 for idx in range(len(or_preds_a)) if a_labels[idx]==0 and or_preds_a[idx]==1)

print(f'\n  APTOS OR-gate (DN121bin + EffNet_bin): Cost={or_cost_a}, SevMiss={or_sev_a}, FP={or_fp_a}')

matching_a = [r for r in sweep_results_a if r['sev'] <= or_sev_a]
if matching_a:
    best_match_a = min(matching_a, key=lambda r: r['cost'])
    print(f'  Best matching single threshold: t={best_match_a["t"]:.2f}, Cost={best_match_a["cost"]}, FP={best_match_a["fp"]}')
    print(f'  Ensemble advantage: cost {best_match_a["cost"] - or_cost_a:+d}, FP {best_match_a["fp"] - or_fp_a:+d}')
else:
    print(f'  No single-model threshold achieves {or_sev_a} or fewer severe misses!')

# ============================================================
print(f'\n{"="*80}')
print('ANALYSIS 2: 2x2 SEPARATION (Metric Effect vs Rule Effect)')
print('='*80)

# For top 10 ensembles, compute all 4 cells of the 2x2
model_names = list(eyepacs_preds.keys())
best_solo_cost = severity_cost((best_probs_e >= 0.5).astype(int), e_labels, eyepacs_grades)

print(f'\n  Best single model ({best_name}) at t=0.5: AUC={eyepacs_preds[best_name]["auc"]:.4f}, Cost={best_solo_cost}')

print(f'\n  {"Ensemble":<45} {"Avg+AUC":<10} {"Avg+Cost":<10} {"OR+Sens":<10} {"OR+Cost":<10}')
print(f'  {"-"*85}')

twobytwo_results = []
for i, n1 in enumerate(model_names):
    for j, n2 in enumerate(model_names):
        if j <= i: continue
        p1 = eyepacs_preds[n1]['probs']
        p2 = eyepacs_preds[n2]['probs']

        # Cell 1: Mean averaging + AUC
        avg_probs = (p1 + p2) / 2
        avg_auc = roc_auc_score(e_labels, avg_probs)

        # Cell 2: Mean averaging + severity cost (threshold 0.5 on averaged probs)
        avg_preds = (avg_probs >= 0.5).astype(int)
        avg_cost = severity_cost(avg_preds, e_labels, eyepacs_grades)

        # Cell 3: OR-gate + sensitivity (clinical discrimination)
        or_preds = np.maximum((p1 >= 0.5).astype(int), (p2 >= 0.5).astype(int))
        or_sens = sum(1 for idx in range(len(or_preds)) if e_labels[idx]==1 and or_preds[idx]==1) / sum(e_labels)

        # Cell 4: OR-gate + severity cost
        or_cost = severity_cost(or_preds, e_labels, eyepacs_grades)

        twobytwo_results.append({
            'n1': n1, 'n2': n2,
            'avg_auc': avg_auc, 'avg_cost': avg_cost,
            'or_sens': or_sens, 'or_cost': or_cost
        })

# Sort by OR cost and show top 15
for r in sorted(twobytwo_results, key=lambda x: x['or_cost'])[:15]:
    name = f"{r['n1'][:15]}+{r['n2'][:15]}"
    print(f'  {name:<45} {r["avg_auc"]:<10.4f} {r["avg_cost"]:<10d} {r["or_sens"]:<10.3f} {r["or_cost"]:<10d}')

# Show the decomposition clearly
print(f'\n  DECOMPOSITION: What drives the effect?')
best_avg_cost = severity_cost((best_probs_e >= 0.5).astype(int), e_labels, eyepacs_grades)
print(f'  Solo model (t=0.5): AUC={eyepacs_preds[best_name]["auc"]:.4f}, Cost={best_avg_cost}')
top = sorted(twobytwo_results, key=lambda x: x['or_cost'])[0]
print(f'  Best ensemble avg+cost: {top["avg_cost"]} (change from solo: {top["avg_cost"] - best_avg_cost:+d})')
print(f'  Best ensemble OR+cost:  {top["or_cost"]} (change from solo: {top["or_cost"] - best_avg_cost:+d})')
print(f'  -> Metric effect (avg cost vs solo cost): {top["avg_cost"] - best_avg_cost:+d}')
print(f'  -> Rule effect (OR cost vs avg cost):     {top["or_cost"] - top["avg_cost"]:+d}')
print(f'  -> Total effect (OR cost vs solo cost):   {top["or_cost"] - best_avg_cost:+d}')

# ============================================================
print(f'\n{"="*80}')
print('ANALYSIS 3: SEVERITY WEIGHT ROBUSTNESS')
print('='*80)

# Sweep grade 4 weight from 50 to 2000, grade 3 from 25 to 500
# Check if top ensemble ranking is stable

weight_scenarios = [
    ('Conservative (100:50:5:1)', {4: 100, 3: 50, 2: 5, 1: 1, 'fp': 1}),
    ('Moderate (250:100:10:1)', {4: 250, 3: 100, 2: 10, 1: 1, 'fp': 1}),
    ('Paper (500:200:15:1)', {4: 500, 3: 200, 2: 15, 1: 2, 'fp': 1}),
    ('Aggressive (1000:400:30:1)', {4: 1000, 3: 400, 2: 30, 1: 2, 'fp': 1}),
    ('Extreme (2000:800:50:1)', {4: 2000, 3: 800, 2: 50, 1: 2, 'fp': 1}),
    ('Equal weight (1:1:1:1)', {4: 1, 3: 1, 2: 1, 1: 1, 'fp': 1}),
    ('Binary only (1:1:0:0:1)', {4: 1, 3: 1, 2: 0, 1: 0, 'fp': 1}),
]

print(f'\n  Does the top ensemble change under different severity weights?')
print(f'  {"Scenario":<30} {"Best Solo Cost":<15} {"Best OR Ensemble":<45} {"OR Cost":<10} {"Reduction":<10}')
print(f'  {"-"*110}')

for scenario_name, w in weight_scenarios:
    # Best single model cost under this weighting
    solo_cost = severity_cost((best_probs_e >= 0.5).astype(int), e_labels, eyepacs_grades, w)

    # Find best OR-gate ensemble under this weighting
    best_or = None
    best_or_cost = float('inf')
    for i, n1 in enumerate(model_names):
        for j, n2 in enumerate(model_names):
            if j <= i: continue
            p1 = eyepacs_preds[n1]['probs']
            p2 = eyepacs_preds[n2]['probs']
            or_preds = np.maximum((p1 >= 0.5).astype(int), (p2 >= 0.5).astype(int))
            cost = severity_cost(or_preds, e_labels, eyepacs_grades, w)
            if cost < best_or_cost:
                best_or_cost = cost
                best_or = f'{n1} + {n2}'

    reduction = (solo_cost - best_or_cost) / solo_cost * 100 if solo_cost > 0 else 0
    print(f'  {scenario_name:<30} {solo_cost:<15d} {best_or:<45} {best_or_cost:<10d} {reduction:<10.1f}%')

# ============================================================
print(f'\n{"="*80}')
print('ANALYSIS 4: BOOTSTRAP CONFIDENCE INTERVALS')
print('='*80)

n_bootstrap = 1000
np.random.seed(42)

# Bootstrap the key quantities for the best ensemble
p1_e = eyepacs_preds['densenet121_5class']['probs']
p2_e = eyepacs_preds['efficientnet_b3_5class']['probs']
n_test = len(e_labels)

boot_solo_cost = []
boot_or_cost = []
boot_rho = []
boot_concordant = []

for _ in range(n_bootstrap):
    idx = np.random.choice(n_test, n_test, replace=True)

    # Solo
    solo_preds = (p1_e[idx] >= 0.5).astype(int)
    sc = severity_cost(solo_preds, e_labels[idx], eyepacs_grades[idx])
    boot_solo_cost.append(sc)

    # OR-gate
    or_preds = np.maximum((p1_e[idx] >= 0.5).astype(int), (p2_e[idx] >= 0.5).astype(int))
    oc = severity_cost(or_preds, e_labels[idx], eyepacs_grades[idx])
    boot_or_cost.append(oc)

    # Rho
    e1 = ((p1_e[idx] >= 0.5).astype(int) != e_labels[idx]).astype(int)
    e2 = ((p2_e[idx] >= 0.5).astype(int) != e_labels[idx]).astype(int)
    if e1.std() > 0 and e2.std() > 0:
        boot_rho.append(np.corrcoef(e1, e2)[0, 1])

    # Concordant severe misses
    severe = eyepacs_grades[idx] >= 3
    both_miss = sum(1 for i2 in range(len(idx)) if severe[i2] and e_labels[idx[i2]]==1
                    and (p1_e[idx[i2]] < 0.5) and (p2_e[idx[i2]] < 0.5))
    boot_concordant.append(both_miss)

print(f'\n  Bootstrap 95% CIs (n={n_bootstrap}) for DN121_5c + EffNet_5c on EyePACS:')
print(f'  Solo severity cost:     {np.mean(boot_solo_cost):.0f} [{np.percentile(boot_solo_cost, 2.5):.0f}, {np.percentile(boot_solo_cost, 97.5):.0f}]')
print(f'  OR-gate severity cost:  {np.mean(boot_or_cost):.0f} [{np.percentile(boot_or_cost, 2.5):.0f}, {np.percentile(boot_or_cost, 97.5):.0f}]')
print(f'  Cost reduction:         {np.mean(np.array(boot_solo_cost) - np.array(boot_or_cost)):.0f} [{np.percentile(np.array(boot_solo_cost) - np.array(boot_or_cost), 2.5):.0f}, {np.percentile(np.array(boot_solo_cost) - np.array(boot_or_cost), 97.5):.0f}]')
print(f'  Error correlation (rho): {np.mean(boot_rho):.4f} [{np.percentile(boot_rho, 2.5):.4f}, {np.percentile(boot_rho, 97.5):.4f}]')
print(f'  Concordant severe miss: {np.mean(boot_concordant):.1f} [{np.percentile(boot_concordant, 2.5):.0f}, {np.percentile(boot_concordant, 97.5):.0f}]')

# Also bootstrap the rho-concordant-miss correlation
boot_r = []
all_model_names = list(eyepacs_preds.keys())
for _ in range(500):
    idx = np.random.choice(n_test, n_test, replace=True)
    pair_rhos = []
    pair_bms = []
    for i, n1 in enumerate(all_model_names):
        for j, n2 in enumerate(all_model_names):
            if j <= i: continue
            e1 = ((eyepacs_preds[n1]['probs'][idx] >= 0.5).astype(int) != e_labels[idx]).astype(int)
            e2 = ((eyepacs_preds[n2]['probs'][idx] >= 0.5).astype(int) != e_labels[idx]).astype(int)
            if e1.std() > 0 and e2.std() > 0:
                rho = np.corrcoef(e1, e2)[0, 1]
            else:
                continue
            severe = eyepacs_grades[idx] >= 3
            bm = sum(1 for i2 in range(len(idx)) if severe[i2] and e_labels[idx[i2]]==1
                     and (eyepacs_preds[n1]['probs'][idx[i2]] < 0.5) and (eyepacs_preds[n2]['probs'][idx[i2]] < 0.5))
            pair_rhos.append(rho)
            pair_bms.append(bm)
    if len(pair_rhos) > 2:
        boot_r.append(np.corrcoef(pair_rhos, pair_bms)[0, 1])

print(f'\n  rho vs concordant miss correlation: {np.mean(boot_r):.4f} [{np.percentile(boot_r, 2.5):.4f}, {np.percentile(boot_r, 97.5):.4f}]')

# ============================================================
print(f'\n{"="*80}')
print('ALL ANALYSES COMPLETE')
print('='*80)

# Save everything
results = {
    'threshold_sweep_eyepacs': sweep_results_e,
    'threshold_sweep_aptos': [dict(r) for r in sweep_results_a],
    'weight_robustness': 'See console output',
    'bootstrap': {
        'solo_cost_ci': [float(np.percentile(boot_solo_cost, 2.5)), float(np.percentile(boot_solo_cost, 97.5))],
        'or_cost_ci': [float(np.percentile(boot_or_cost, 2.5)), float(np.percentile(boot_or_cost, 97.5))],
        'rho_ci': [float(np.percentile(boot_rho, 2.5)), float(np.percentile(boot_rho, 97.5))],
        'rho_bothmiss_r_ci': [float(np.percentile(boot_r, 2.5)), float(np.percentile(boot_r, 97.5))],
    }
}

with open(f'{RESULTS_DIR}/npj_robustness_results.json', 'w') as f:
    json.dump(results, f, indent=2, default=str)
print(f'\nSaved to {RESULTS_DIR}/npj_robustness_results.json')

## Summary

All results are saved to `results_eyepacs_full/` on Google Drive:
- `comprehensive_evaluation.json` — Full 55-pair analysis on both datasets
- `npj_robustness_results.json` — Threshold sweep, bootstrap CIs, weight robustness

These files support the NPJ Digital Medicine manuscript.